In [1]:
import importlib
import os
import platform
import sys
from pathlib import Path

print(f"Python version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")
print(f"Working folder : {Path.cwd()}")

REQUIRED_PACKAGES = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("yaml", "pyyaml"),
    ("joblib", "joblib"),
    ("pydantic", "pydantic"),
    ("dotenv", "python-dotenv"),
]

missing = []
for import_name, pip_name in REQUIRED_PACKAGES:
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, "__version__", "installed")
        print(f"  [PASS] {pip_name:<15} -> {version}")
    except ImportError:
        print(f"  [FAIL] {pip_name:<15} -> NOT INSTALLED")
        missing.append(pip_name)

assert not missing, f"Missing packages: {missing}. Run pip install -r ../docker-jupyterlab/requirements.txt"
print("\nAll required libraries imported successfully!")

Python version : 3.11.18
Platform       : Linux-6.18.40.1-microsoft-standard-WSL2-x86_64-with-glibc2.41
Working folder : /workspace/notebooks
  [PASS] numpy           -> 2.1.3
  [PASS] pandas          -> 2.2.3
  [PASS] scikit-learn    -> 1.6.1
  [PASS] matplotlib      -> 3.10.0
  [PASS] seaborn         -> 0.13.2
  [PASS] pyyaml          -> 6.0.2
  [PASS] joblib          -> 1.4.2
  [PASS] pydantic        -> 2.10.6
  [PASS] python-dotenv   -> installed

All required libraries imported successfully!


In [2]:
#############################################################################################################################
# Ensure data files exist (auto-generate cleanly if missing)
data_dir = Path("data")
expected_files = [
    data_dir / "customers_raw.csv",
    data_dir / "usage_events.json",
    data_dir / "customers_corrupted_batch.csv",
    data_dir / "incoming_batch_exercise.csv",
]

if not all(p.exists() for p in expected_files):
    print("Some data files are missing — running data/generate_data.py...")
    from data.generate_data import generate_all_datasets
    generate_all_datasets(data_dir)

for p in expected_files:
    assert p.exists(), f"Missing dataset: {p}"
    print(f"  [PASS] Found {p} ({p.stat().st_size:,} bytes)")

config_path = Path("configs/train_config.yaml")
assert config_path.exists(), f"Missing config: {config_path}"
print(f"  [PASS] Found {config_path} ({config_path.stat().st_size:,} bytes)")

  [PASS] Found data/customers_raw.csv (52,225 bytes)
  [PASS] Found data/usage_events.json (168,553 bytes)
  [PASS] Found data/customers_corrupted_batch.csv (2,148 bytes)
  [PASS] Found data/incoming_batch_exercise.csv (7,726 bytes)
  [PASS] Found configs/train_config.yaml (1,534 bytes)


In [3]:
# Verify write access to artifacts/ folder
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
probe_file = artifacts_dir / ".smoke_test_ok"
probe_file.write_text("Day 6 environment check passed!\n", encoding="utf-8")
assert probe_file.exists()
print(f"  [PASS] Write check succeeded at {probe_file.resolve()}")
print("\nYou are 100% ready for Day 6! Raise your green sticky note.")

  [PASS] Write check succeeded at /workspace/notebooks/artifacts/.smoke_test_ok

You are 100% ready for Day 6! Raise your green sticky note.


In [4]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd
from pydantic import BaseModel, Field, ValidationError

RAW_CUSTOMERS_PATH = Path("data/customers_raw.csv")
RAW_USAGE_PATH = Path("data/usage_events.json")
CORRUPTED_PATH = Path("data/customers_corrupted_batch.csv")

In [5]:
raw_customers = pd.read_csv(RAW_CUSTOMERS_PATH)
with open(RAW_USAGE_PATH, "r", encoding="utf-8") as f:
    raw_usage = pd.DataFrame(json.load(f))

print(f"Raw customers rows : {len(raw_customers)} (unique customer_id: {raw_customers['customer_id'].nunique()})")
print(f"Raw usage rows     : {len(raw_usage)} (unique customer_id: {raw_usage['customer_id'].nunique()})")
print("\nRaw plan_tier values (notice messy casing & spaces!):")
print(raw_customers["plan_tier"].value_counts(dropna=False))
print("\nRaw country values:")
print(raw_customers["country"].value_counts(dropna=False))

Raw customers rows : 1003 (unique customer_id: 1000)
Raw usage rows     : 1001 (unique customer_id: 1001)

Raw plan_tier values (notice messy casing & spaces!):
plan_tier
basic        452
standard     331
premium      214
NaN            3
 Basic         1
PREMIUM        1
 Standard      1
Name: count, dtype: int64

Raw country values:
country
US      360
IN      297
UK      144
DE      102
CA       98
 us       1
in        1
Name: count, dtype: int64


In [6]:
def compute_file_sha256(path: Path) -> str:
    """Return a short SHA-256 fingerprint for data lineage tracking."""
    h = hashlib.sha256()
    h.update(path.read_bytes())
    return h.hexdigest()[:12]


def ingest_streampulse_data(customers_csv: Path, usage_json: Path) -> tuple[pd.DataFrame, dict]:
    """Load, standardize, deduplicate, and join StreamPulse customer + usage data."""
    cust_df = pd.read_csv(customers_csv)
    with open(usage_json, "r", encoding="utf-8") as f:
        usage_df = pd.DataFrame(json.load(f))

    raw_row_count = len(cust_df)

    # 1. Standardize column names (lowercase, stripped)
    cust_df.columns = [c.strip().lower() for c in cust_df.columns]
    usage_df.columns = [c.strip().lower() for c in usage_df.columns]

    # 2. Clean whitespace & casing on string categories (preserving NaN as NaN!)
    if "plan_tier" in cust_df.columns:
        cust_df["plan_tier"] = cust_df["plan_tier"].apply(
            lambda x: x.strip().lower() if isinstance(x, str) else x
        )
    if "country" in cust_df.columns:
        cust_df["country"] = cust_df["country"].apply(
            lambda x: x.strip().upper() if isinstance(x, str) else x
        )
    if "payment_method" in cust_df.columns:
        cust_df["payment_method"] = cust_df["payment_method"].apply(
            lambda x: x.strip().lower() if isinstance(x, str) else x
        )

    # 3. Deduplicate on primary key customer_id
    duplicates_removed = int(cust_df.duplicated(subset=["customer_id"]).sum())
    cust_df = cust_df.drop_duplicates(subset=["customer_id"], keep="first")

    # 4. Left join usage metrics onto customers (drop orphan usage IDs with no billing profile)
    merged_df = cust_df.merge(usage_df, on="customer_id", how="left", validate="one_to_one")

    metadata = {
        "ingested_at_utc": datetime.now(timezone.utc).isoformat(),
        "customers_source": str(customers_csv),
        "customers_sha256": compute_file_sha256(customers_csv),
        "usage_source": str(usage_json),
        "usage_sha256": compute_file_sha256(usage_json),
        "raw_customer_rows": raw_row_count,
        "duplicates_removed": duplicates_removed,
        "final_rows": len(merged_df),
        "final_columns": list(merged_df.columns),
    }
    return merged_df, metadata


ingested_df, ingest_meta = ingest_streampulse_data(RAW_CUSTOMERS_PATH, RAW_USAGE_PATH)
print("Ingestion Metadata:")
print(json.dumps(ingest_meta, indent=2))
print("\nCleaned plan_tier values:")
print(ingested_df["plan_tier"].value_counts(dropna=False))

Ingestion Metadata:
{
  "ingested_at_utc": "2026-11-20T07:05:31.686323+00:00",
  "customers_source": "data/customers_raw.csv",
  "customers_sha256": "f5bc3e6b1a1a",
  "usage_source": "data/usage_events.json",
  "usage_sha256": "d2c0a2fdecb3",
  "raw_customer_rows": 1003,
  "duplicates_removed": 3,
  "final_rows": 1000,
  "final_columns": [
    "customer_id",
    "age",
    "plan_tier",
    "country",
    "payment_method",
    "auto_pay",
    "tenure_months",
    "monthly_charges",
    "churned",
    "login_days_30d",
    "watch_hours_30d",
    "support_tickets_30d",
    "last_active_device"
  ]
}

Cleaned plan_tier values:
plan_tier
basic       452
standard    330
premium     215
NaN           3
Name: count, dtype: int64


In [7]:
REQUIRED_COLUMNS = [
    "customer_id", "age", "plan_tier", "country", "payment_method",
    "auto_pay", "tenure_months", "monthly_charges", "churned",
]
ALLOWED_PLANS = {"basic", "standard", "premium"}
ALLOWED_COUNTRIES = {"US", "IN", "UK", "CA", "DE"}


def validate_customer_dataframe(df: pd.DataFrame, max_missing_ratio: float = 0.15) -> dict:
    """Run schema and data-quality checks. Returns a structured validation report."""
    errors = []
    warnings = []

    # Check 1: Required columns
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_cols:
        errors.append(f"Missing required columns: {missing_cols}")
        return {"passed": False, "row_count": len(df), "errors": errors, "warnings": warnings}

    # Check 2: Primary key uniqueness & non-null
    null_ids = int(df["customer_id"].isna().sum())
    dup_ids = int(df["customer_id"].duplicated().sum())
    if null_ids > 0:
        errors.append(f"Found {null_ids} null customer_id values.")
    if dup_ids > 0:
        errors.append(f"Found {dup_ids} duplicate customer_id values.")

    # Check 3: Missingness threshold per column
    for col in REQUIRED_COLUMNS:
        ratio = float(df[col].isna().mean())
        if ratio > max_missing_ratio:
            errors.append(f"Column '{col}' missing ratio {ratio:.1%} exceeds limit {max_missing_ratio:.1%}.")
        elif ratio > 0:
            warnings.append(f"Column '{col}' has {ratio:.1%} missing values (will be imputed in pipeline).")

    # Check 4: Numeric range checks (ignoring NaNs which are checked above)
    bad_age = df["age"].dropna().pipe(lambda s: ~s.between(18, 100)).sum()
    if bad_age > 0:
        errors.append(f"Found {int(bad_age)} rows with age outside [18, 100].")

    bad_tenure = (df["tenure_months"].dropna() < 0).sum()
    if bad_tenure > 0:
        errors.append(f"Found {int(bad_tenure)} rows with negative tenure_months.")

    bad_charges = df["monthly_charges"].dropna().pipe(lambda s: ~s.between(0, 100)).sum()
    if bad_charges > 0:
        errors.append(f"Found {int(bad_charges)} rows with monthly_charges outside [0, 100].")

    # Check 5: Allowed categorical values
    bad_plans = set(df["plan_tier"].dropna().unique()) - ALLOWED_PLANS
    if bad_plans:
        errors.append(f"Unexpected plan_tier values: {sorted(bad_plans)}")

    bad_countries = set(df["country"].dropna().unique()) - ALLOWED_COUNTRIES
    if bad_countries:
        errors.append(f"Unexpected country values: {sorted(bad_countries)}")

    # Check 6: Target label integrity
    bad_targets = set(df["churned"].dropna().unique()) - {0, 1}
    if bad_targets:
        errors.append(f"Invalid churned target values: {sorted(bad_targets)}")

    return {
        "passed": len(errors) == 0,
        "row_count": int(len(df)),
        "error_count": len(errors),
        "errors": errors,
        "warnings": warnings,
    }


clean_report = validate_customer_dataframe(ingested_df)
print("=== Validation Report: Clean Ingested Dataset ===")
print(json.dumps(clean_report, indent=2))
assert clean_report["passed"] is True

=== Validation Report: Clean Ingested Dataset ===
{
  "passed": true,
  "row_count": 1000,
  "error_count": 0,
  "errors": [],
  "warnings": [
    "Column 'age' has 0.5% missing values (will be imputed in pipeline).",
    "Column 'plan_tier' has 0.3% missing values (will be imputed in pipeline).",
    "Column 'monthly_charges' has 0.4% missing values (will be imputed in pipeline)."
  ]
}


In [8]:
corrupted_df = pd.read_csv(CORRUPTED_PATH)
corrupted_report = validate_customer_dataframe(corrupted_df)

print("=== Validation Report: Corrupted Batch ===")
print(json.dumps(corrupted_report, indent=2))
assert corrupted_report["passed"] is False

# Save validation report artifact for auditability
report_path = Path("artifacts/validation_report_demo.json")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(corrupted_report, indent=2), encoding="utf-8")
print(f"\nSaved audit report to: {report_path}")

=== Validation Report: Corrupted Batch ===
{
  "passed": false,
  "row_count": 40,
  "error_count": 5,
  "errors": [
    "Found 2 rows with age outside [18, 100].",
    "Found 1 rows with negative tenure_months.",
    "Found 1 rows with monthly_charges outside [0, 100].",
    "Unexpected plan_tier values: [' Basic ', 'ultra_vip']",
    "Invalid churned target values: [np.int64(2)]"
  ],
  "warnings": [
    "Column 'age' has 5.0% missing values (will be imputed in pipeline).",
    "Column 'plan_tier' has 2.5% missing values (will be imputed in pipeline).",
    "Column 'monthly_charges' has 2.5% missing values (will be imputed in pipeline)."
  ]
}

Saved audit report to: artifacts/validation_report_demo.json


In [9]:
class CustomerRecordSchema(BaseModel):
    customer_id: str
    age: float = Field(ge=18, le=100)
    plan_tier: str = Field(pattern="^(basic|standard|premium)$")
    tenure_months: float = Field(ge=0)
    monthly_charges: float = Field(ge=0, le=100)


good_payload = {
    "customer_id": "CUST-50001",
    "age": 31,
    "plan_tier": "standard",
    "tenure_months": 14,
    "monthly_charges": 15.99,
}
bad_payload = {
    "customer_id": "CUST-50002",
    "age": -5,
    "plan_tier": "ultra_vip",
    "tenure_months": 14,
    "monthly_charges": 999.0,
}

print("Valid payload accepted:", CustomerRecordSchema(**good_payload))
print("\nTesting invalid payload:")
try:
    CustomerRecordSchema(**bad_payload)
except ValidationError as exc:
    for err in exc.errors():
        print(f"  - Field '{err['loc'][0]}': {err['msg']} (got {err['input']!r})")

Valid payload accepted: customer_id='CUST-50001' age=31.0 plan_tier='standard' tenure_months=14.0 monthly_charges=15.99

Testing invalid payload:
  - Field 'age': Input should be greater than or equal to 18 (got -5)
  - Field 'plan_tier': String should match pattern '^(basic|standard|premium)$' (got 'ultra_vip')
  - Field 'monthly_charges': Input should be less than or equal to 100 (got 999.0)


In [10]:
def add_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create stateless, row-level domain features for StreamPulse churn prediction."""
    out = df.copy()
    # Add +1 in denominators to prevent division-by-zero when login_days_30d == 0
    out["charges_per_login"] = out["monthly_charges"] / (out["login_days_30d"] + 1.0)
    out["tickets_per_tenure_month"] = out["support_tickets_30d"] / (out["tenure_months"] + 1.0)
    out["watch_hours_per_login"] = out["watch_hours_30d"] / (out["login_days_30d"] + 1.0)
    out["is_new_customer"] = (out["tenure_months"] <= 3).astype(int)
    return out


df_fe = add_domain_features(df)

# Quick sanity check: compare average engineered features for Churned vs Retained customers!
check_cols = ["charges_per_login", "tickets_per_tenure_month", "watch_hours_per_login", "is_new_customer"]
print("Mean engineered features grouped by churned (0 = Retained, 1 = Churned):")
df_fe.groupby("churned")[check_cols].mean().round(3)

NameError: name 'df' is not defined